# Gridless PCB Router — Colab Training Notebook

Trains `DualStreamRouter` with PPO on curriculum PCB boards, with live plots and board snapshots each epoch, and checkpoints saved to **your Google Drive** so training survives Colab disconnects.

Run the cells in order, top to bottom. If Colab disconnects or you close the tab, just reopen this notebook and re-run all cells — training resumes from the last checkpoint automatically (see "Resuming after a disconnect" near the bottom).

Background: [PROJECT.md](https://github.com/Klutzhehe/Router-v2/blob/main/PROJECT.md) has the full architecture writeup; [docs/reward-function.md](https://github.com/Klutzhehe/Router-v2/blob/main/docs/reward-function.md) has the reward math.

## Step 1 — Mount Drive and get the code

Clones the repo into your Drive the first time; every run after that just `git pull`s so you get the latest code without re-cloning.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

REPO_URL = "https://github.com/Klutzhehe/Router-v2.git"
REPO_DIR = "/content/drive/MyDrive/Router-v2"
CKPT_DIR = "/content/drive/MyDrive/Router-v2-checkpoints"

import os
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} "{REPO_DIR}"
else:
    !cd "{REPO_DIR}" && git pull

os.makedirs(CKPT_DIR, exist_ok=True)
print("repo:", REPO_DIR)
print("checkpoints:", CKPT_DIR)

In [ ]:
import sys
sys.path.insert(0, f"{REPO_DIR}/python")

import json
import time
from collections import deque
from pathlib import Path

import numpy as np
import torch

from pcb_router.config import EnvConfig
from pcb_router.env import RoutingEnv
from pcb_router.generator import STAGES, generate_board
from pcb_router.model import DualStreamRouter
from pcb_router.ppo import PPO, PPOConfig, collect_rollout, load_checkpoint, save_checkpoint, to_torch
from pcb_router.viz import TrainingMonitor, show_board_inline

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
for i, s in enumerate(STAGES):
    print(f"  stage {i}: {s.n_nets} nets, {s.layers} layers, {s.size}x{s.size}mm board")

## Step 2 — Configuration

`RUN_NAME` names the checkpoint (`{RUN_NAME}.pt`) and log file (`{RUN_NAME}_history.jsonl`) in `CKPT_DIR`. Use a new `RUN_NAME` to start a fresh, independent run; reuse one to resume it.

The curriculum auto-advances to the next stage once the rolling 20-episode completion rate clears `ADVANCE_AT`. `STAGE` below is only the **starting** stage — if a checkpoint already exists for this `RUN_NAME`, its saved stage/step-count/optimizer-state take over and this value is ignored.

In [ ]:
RUN_NAME = "router"          # checkpoint file: {CKPT_DIR}/{RUN_NAME}.pt
STAGE = 0                    # starting curriculum stage (ignored if resuming)
TOTAL_STEPS = 500_000        # stop after this many env steps (cumulative across resumes)
ROLLOUT_STEPS = 2048         # env steps per PPO update = one "epoch" below
ADVANCE_AT = 0.95            # rolling completion rate needed to advance curriculum
RENDER_EVERY = 5             # show a board snapshot every N epochs
SEED = 0

ckpt_path = Path(CKPT_DIR) / f"{RUN_NAME}.pt"
log_path = Path(CKPT_DIR) / f"{RUN_NAME}_history.jsonl"
torch.manual_seed(SEED)

## Step 3 — Build the model and resume if a checkpoint exists

In [ ]:
model = DualStreamRouter().to(device)
ppo = PPO(model, PPOConfig(), device=device)

stage = STAGE
steps_done = 0
completions = deque(maxlen=20)
history = []

if ckpt_path.exists():
    ckpt = load_checkpoint(ckpt_path, model, ppo, device=device)
    stage = ckpt.get("stage", STAGE)
    steps_done = ckpt.get("steps_done", 0)
    completions.extend(ckpt.get("completions", []))
    print(f"resumed '{RUN_NAME}': stage={stage}  steps_done={steps_done:,}  "
          f"rolling completion={np.mean(completions) if completions else 0:.1%}")
else:
    print(f"no checkpoint at {ckpt_path} -- starting a fresh run at stage {stage}")

# Replay the on-disk history log (if any) into the live plot so the monitor
# picks up right where the last session left off instead of starting blank.
monitor = TrainingMonitor(advance_at=ADVANCE_AT)
if log_path.exists():
    with open(log_path) as f:
        history = [json.loads(line) for line in f]

env = RoutingEnv(lambda rng, s=stage: generate_board(s, rng), cfg=EnvConfig(), seed=SEED + stage)
carried = (None, None)
print(f"stage {stage}: {STAGES[min(stage, len(STAGES)-1)].n_nets} nets, "
      f"{STAGES[min(stage, len(STAGES)-1)].layers} layers")

## Expected behavior per curriculum stage, and what to do if it doesn't happen

**Measured baseline** (this exact codebase, 49k steps, CPU, stage 0): rolling completion rate went **4.8% → 35.0%**, mean episode return **−115 → −48**, entropy **4.92 → 4.25**, and it was still climbing at cutoff. Everything below stage 0 is an *extrapolation* from that single data point, not a separate measurement — treat the step counts as rough budgets, not guarantees, and use the diagnostics (not the clock) to decide if something is actually wrong.

| Stage | Board | Rough step budget to reach ~90%+ completion | Notes |
|---|---|---|---|
| 0 | 3 nets, 2 layers, 20×20mm | 100k–400k | Simplest possible case. If this stage can't clear ~90% within ~500k steps, the pipeline itself likely has a bug — don't proceed to bigger boards, debug here first. |
| 1 | 6 nets, 2 layers, 25×25mm | 200k–600k | Adds keep-outs; congestion-avoidance starts to matter. |
| 2 | 8 nets, 2 layers, +cross-layer pads | 300k–800k | First stage that *requires* via usage to complete some nets. |
| 3 | 12 nets, 4 layers, 30×30mm | 500k–1.5M | More layers = larger via/layer action space. |
| 4 | 20 nets, 6 layers, 40×40mm | 1M–3M | |
| 5 | 30 nets, 12 layers, 50×50mm | 2M+ | Realistically may need the imitation-warm-start roadmap item (PROJECT.md §8) rather than pure PPO from scratch. |

### Diagnostics — read these off the live plot each time you check in

- **DRC count must always be 0.** Nothing above depends on it, because it *should* be structurally impossible (actions are legal by construction — see PROJECT.md). If you ever see `info["drc"] > 0` in a rollout, **stop training** and report it — that's a geometry-kernel bug, not a hyperparameter issue.
- **Entropy collapses toward 0 within the first ~10–20 epochs, before completion rises much.** → premature convergence / exploration collapse. Fix: raise `PPOConfig.ent_coef` (currently 0.01) to 0.02–0.03, or lower `lr`.
- **Entropy stays high/flat *and* completion rate is flat *and* `pi_loss` hovers near 0 for many epochs.** → no learning signal is reaching the policy. Check the reward isn't degenerate for this board (e.g. all episodes end in immediate failure), and that advantages aren't collapsing to ~0 (print `buf.adv.std()` in `ppo.py:update` if unsure).
- **Value loss (`v_loss`) grows without bound instead of settling.** → lower `lr`, or the reward scale changed (check `RewardWeights` in `config.py` wasn't edited without re-checking `docs/reward-function.md`).
- **Completion rate genuinely plateaus** (flat for 100k+ steps, not still trending up) well below the stage's rough budget above. → this is the expected failure mode for pure RL-from-scratch on harder boards; see PROJECT.md §8 item 2 (A\* teacher + imitation warm-start) before spending more compute on more PPO steps.
- **steps/s drops noticeably over a long session.** → usually Drive I/O latency from saving a checkpoint every single epoch, not a code leak. If it's a problem, raise the checkpoint cadence (edit the loop below to save every 3rd epoch instead of every epoch).
- **Something is unrecoverably wrong and you want to restart this stage from scratch.** → delete `{RUN_NAME}.pt` and `{RUN_NAME}_history.jsonl` from `CKPT_DIR` (or just change `RUN_NAME`) and re-run from Step 3.

## Step 4 — Train

One "epoch" here = one PPO rollout (`ROLLOUT_STEPS` env steps) + one PPO update. Each epoch: the plot redraws, the checkpoint + history log save to Drive, and every `RENDER_EVERY` epochs a board snapshot from the current policy is shown inline.

**Safe to interrupt at any time** (Colab "Interrupt execution", or just closing the tab): the last completed epoch's checkpoint is already on Drive. Re-running this cell continues from there — it does not restart the epoch that was interrupted mid-flight, only from the last one that finished.

In [ ]:
try:
    while steps_done < TOTAL_STEPS:
        t0 = time.time()
        buf, stats, carried = collect_rollout(env, model, ROLLOUT_STEPS, device, *carried)
        upd = ppo.update(buf)
        steps_done += ROLLOUT_STEPS
        completions.extend(stats["completions"])

        record = {
            "steps_done": steps_done, "stage": stage,
            "ep_return": float(np.mean(stats["returns"])) if stats["returns"] else float("nan"),
            "completion": float(np.mean(completions)) if completions else 0.0,
            "steps_per_sec": ROLLOUT_STEPS / (time.time() - t0),
            **upd,
        }
        history.append(record)
        with open(log_path, "a") as f:
            f.write(json.dumps(record) + "\n")
        save_checkpoint(ckpt_path, model, ppo, stage, steps_done, completions, history)

        epoch = len(history)
        if epoch % RENDER_EVERY == 0:
            demo_env = RoutingEnv(lambda rng, s=stage: generate_board(s, rng), cfg=EnvConfig(),
                                  seed=int(np.random.default_rng().integers(1_000_000)))
            d_obs, d_masks = demo_env.reset()
            done = False
            with torch.no_grad():
                while not done:
                    t_masks = {k: torch.from_numpy(v).unsqueeze(0).to(device) for k, v in d_masks.items()}
                    a, _, _ = model.act(to_torch(d_obs, device), t_masks, deterministic=True)
                    d_obs, d_masks, _, done, d_info = demo_env.step(
                        (int(a.action_type), int(a.angle_bin), float(a.dist_frac), int(a.layer)))
            show_board_inline(demo_env.board,
                              title=(f"epoch {epoch} | steps {steps_done:,} | stage {stage} | "
                                    f"{d_info['nets_done']}/{d_info['nets_total']} nets | DRC={d_info['drc']}"),
                              completed=demo_env.completed)

        monitor.update(record)
        print(f"steps {steps_done:>8}  stage {stage}  ep_return {record['ep_return']:8.2f}  "
              f"completion {record['completion']:5.1%}  entropy {upd['entropy']:6.3f}  "
              f"pi {upd['pi_loss']:+.4f}  v {upd['v_loss']:8.3f}  {record['steps_per_sec']:6.0f} steps/s")

        if (len(completions) == completions.maxlen and np.mean(completions) >= ADVANCE_AT
                and stage < len(STAGES) - 1):
            stage += 1
            print(f"=== curriculum advance -> stage {stage} ===")
            env = RoutingEnv(lambda rng, s=stage: generate_board(s, rng), cfg=EnvConfig(), seed=SEED + stage)
            completions.clear()
            carried = (None, None)
            save_checkpoint(ckpt_path, model, ppo, stage, steps_done, completions, history)

    print("training complete")
except KeyboardInterrupt:
    print(f"interrupted at step {steps_done:,} -- last checkpoint on Drive is up to date, safe to stop.")

## Resuming after a disconnect

Colab disconnects, or you close the tab, or the runtime recycles. Nothing is lost as long as Drive had synced (it saves every epoch). To pick back up:

1. Reopen this notebook (from Drive, or re-clone/open from GitHub).
2. Run **Step 1** (mount Drive + `git pull` — picks up any code changes pushed to GitHub since your last session).
3. Run **Step 2** with the *same* `RUN_NAME` you used before.
4. Run **Step 3** — it will print `resumed '...'` with the stage/step count it found.
5. Run **Step 4** — training continues exactly where it left off, optimizer momentum and all.

You do **not** need to re-run Step 4 with different `TOTAL_STEPS` math — it's a cumulative target across the whole run, not per-session.

## Extra: render one policy rollout on demand

In [ ]:
demo_env = RoutingEnv(lambda rng, s=stage: generate_board(s, rng), cfg=EnvConfig(), seed=123)
d_obs, d_masks = demo_env.reset()
done = False
with torch.no_grad():
    while not done:
        t_masks = {k: torch.from_numpy(v).unsqueeze(0).to(device) for k, v in d_masks.items()}
        a, _, _ = model.act(to_torch(d_obs, device), t_masks, deterministic=True)
        d_obs, d_masks, _, done, d_info = demo_env.step(
            (int(a.action_type), int(a.angle_bin), float(a.dist_frac), int(a.layer)))
show_board_inline(demo_env.board,
                  title=f"stage {stage} | {d_info['nets_done']}/{d_info['nets_total']} nets | DRC={d_info['drc']}",
                  completed=demo_env.completed)